In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/term-deposit-marketing-2020.csv")

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numerical_cols = [
    "age",
    "balance",
    "day",
    #"duration",
    "campaign"
]

one_hot_encode_cols = [
    "job",
    "marital",
    "education",
    "contact",
    "month"
]

label_encode_cols = [
    "default",
    "housing",
    "loan"
]

In [4]:
from sklearn.model_selection import train_test_split

df["education"] = df["education"].replace("unknown", "missing")
X = df.drop(columns=["y", "duration"])
y = df["y"].map({"no": 0, "yes": 1})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
#feature selection of the original features (not the one-hot encoded features)

import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.base import BaseEstimator, TransformerMixin


class FeatureSelector(BaseEstimator, TransformerMixin):

    def __init__(self, k="all"):
        self.k = k

    def fit(self, X, y):

        # don't modify original data
        X = X.copy()
        X_encoded = X.copy()

        feature_scores = []

        for column in X_encoded:

            # Encode categorical features using label encoding
            if column in label_encode_cols:
                X_encoded[column] = X_encoded[column].astype("category").cat.codes.to_numpy().reshape(-1, 1)
                score = mutual_info_classif(X_encoded[[column]], y, discrete_features=True, random_state=42)[0]

            # Encode categorical features using one-hot encoding
            elif column in one_hot_encode_cols:  
                encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
                X_encoded[column] = encoder.fit_transform(X_encoded[[column]])

                scores = mutual_info_classif(X_encoded[[column]], y, random_state=42)
                score = scores.max()  # Take the maximum score among the one-hot encoded features

            # Numerical feature
            else:
                score = mutual_info_classif(X[[column]],y,random_state=42)[0]

            feature_scores.append(score)

        # Calculate feature scores
        self.scores_ = np.array(feature_scores)

        # Rank features from highest to lowest score
        self.indices_ = np.argsort(self.scores_)[::-1]

        if self.k == "all":
            self.selected_indices_ = self.indices_
        else:
            self.selected_indices_ = self.indices_[:self.k]

        #change selected features to original feature names
        self.selected_features_ = X.columns[self.selected_indices_].tolist()

        return self

    #return dataset with only the selected features
    def transform(self, X):
        return X[self.selected_features_]

    def get_feature_names_out(self, input_features=None):
        return np.array(self.selected_features_)

In [6]:
from sklearn.preprocessing import OrdinalEncoder


class DynamicPreprocessor(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.preprocessor_ = None

    def fit(self, X, y=None):

        # Select only the features that are present in the dataset
        selected_numerical = [col for col in numerical_cols if col in X.columns]
        selected_label = [col for col in label_encode_cols if col in X.columns]
        selected_one_hot = [col for col in one_hot_encode_cols if col in X.columns]

        self.preprocessor_ = ColumnTransformer(
            transformers=[
                ("number", StandardScaler(), selected_numerical),
                ("one_hot", OneHotEncoder(handle_unknown="ignore"), selected_one_hot),
                ("label", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), selected_label)
            ]
        )

        self.preprocessor_.fit(X, y)

        return self

    # return the transformed dataset with only the selected features
    def transform(self, X):
        return self.preprocessor_.transform(X)

    def get_feature_names_out(self, input_features=None):
        return self.preprocessor_.get_feature_names_out(input_features)

In [7]:
#setup baseline models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from xgboost import XGBClassifier


#variables for models

#feature selector
feature_selector = FeatureSelector()

#preprocessor
preprocessor = DynamicPreprocessor()

#scale pos weight for xgboost
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]


models = {
    "Logistic Regression": {
        "pipeline": Pipeline([
            ("select", feature_selector),
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced"))
        ]),
        "params": {
            "model__C": [0.01, 0.1, 1, 10],
            "select__k": [3, 5, 7, 10, "all"]
        }
    }, 

    "XGBoost": {
            "pipeline": Pipeline([
                ("select", feature_selector),
                ("preprocessor", preprocessor),
                ("model", XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight))
            ]),
            "params": {
                "model__max_depth": [1, 2, 3, 4, 6, 10],
                "model__learning_rate": [0.05, 0.1, 0.2],
                "model__n_estimators": [50, 100, 200],
                "select__k": [3, 5, 7, 10, "all"]
            }
        },

    

        "KNN": {
        "pipeline": Pipeline([
            ("select", feature_selector),
            ("preprocessor", preprocessor),
            ("model", KNeighborsClassifier())
        ]),
        "params": {
            "model__n_neighbors": [3, 5, 7, 9, 11],
            "model__weights": ["uniform", "distance"],
            "select__k": [3, 5, 7, 10, "all"]
        }
    }
}
    


In [8]:
from sklearn.model_selection import StratifiedKFold


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [9]:
import time
from sklearn.model_selection import GridSearchCV


results = []

# Metrics to calculate during cross-validation
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

grid_search_results = {}


for name, config in models.items():
    print(f"Training model: {name}")

    # Start timer
    start_time = time.perf_counter()

    # Grid search
    grid_search = GridSearchCV(config["pipeline"], config["params"], cv=cv, scoring=scoring, refit="f1", n_jobs=-1)

    grid_search.fit(X_train, y_train)

    # Store the complete GridSearchCV object
    grid_search_results[name] = grid_search

    best_model = grid_search.best_estimator_

    best_index = grid_search.best_index_

    # Get feature selection information
    selector = best_model.named_steps["select"]

    selected_features = selector.selected_features_
    feature_scores = selector.scores_
    
    elapsed_time = time.perf_counter() - start_time

    # Record results
    results.append({
        "Model": name,
        "Accuracy": grid_search.cv_results_["mean_test_accuracy"][best_index],
        "Precision": grid_search.cv_results_["mean_test_precision"][best_index],
        "Recall": grid_search.cv_results_["mean_test_recall"][best_index],
        "F1 Score": grid_search.cv_results_["mean_test_f1"][best_index],
        "Training Time (seconds)": elapsed_time,
        "Selected Features": selected_features,
        "Best Parameters": grid_search.best_params_
    })


Training model: Logistic Regression
Training model: XGBoost
Training model: KNN


In [10]:
results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

              Model  Accuracy  Precision   Recall  F1 Score  Training Time (seconds)                                                                              Selected Features                                                                                        Best Parameters
Logistic Regression  0.699469   0.126846 0.535179  0.205052                11.954453                                                            [contact, month, age, balance, day]                                                                       {'model__C': 10, 'select__k': 5}
            XGBoost  0.846656   0.190903 0.345686  0.245922               122.403800 [contact, month, age, balance, day, campaign, marital, education, job, housing, loan, default] {'model__learning_rate': 0.05, 'model__max_depth': 10, 'model__n_estimators': 200, 'select__k': 'all'}
                KNN  0.926156   0.460612 0.107471  0.174005                59.379362                                                            [contac

In [11]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

results = []

for name, grid_search in grid_search_results.items():

    # Best pipeline found during GridSearchCV
    final_model = grid_search.best_estimator_

    # Predict ONLY on the untouched test set
    y_pred = final_model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    results.append({
        "Model": name,
        "CV F1": grid_search.best_score_,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Best Parameters": grid_search.best_params_,
        "Confusion Matrix": cm,
    })


In [12]:
#display testing results

results_df = pd.DataFrame(results)
pd.set_option("display.max_columns", None)
print(results_df)

                 Model     CV F1  Accuracy  Precision    Recall  F1 Score  \
0  Logistic Regression  0.205052   0.71700   0.139803  0.564767  0.224126   
1              XGBoost  0.245922   0.84550   0.204320  0.392055  0.268639   
2                  KNN  0.174005   0.92625   0.449541  0.084629  0.142442   

                                     Best Parameters  \
0                   {'model__C': 10, 'select__k': 5}   
1  {'model__learning_rate': 0.05, 'model__max_dep...   
2  {'model__n_neighbors': 5, 'model__weights': 'u...   

             Confusion Matrix  
0  [[5409, 2012], [252, 327]]  
1   [[6537, 884], [352, 227]]  
2     [[7361, 60], [530, 49]]  


In [14]:
#use xgboost to predict segments of clients to prioritize

# Get the best XGBoost model
best_xgb = grid_search_results["XGBoost"].best_estimator_

# Generate probability of subscribing for every customer
test_probabilities = best_xgb.predict_proba(X_test)[:, 1]

# Create a copy of the original customer data
customer_analysis = X_test.copy()

# Add predicted probability
customer_analysis["subscription_probability"] = test_probabilities

# Add actual outcome for comparison
customer_analysis["actual_subscription"] = y_test.values

# Rank customers from highest to lowest probability
customer_analysis = customer_analysis.sort_values("subscription_probability", ascending=False)

customer_analysis.head(20)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,subscription_probability,actual_subscription
24064,33,services,married,secondary,no,3444,yes,no,telephone,21,oct,1,0.986582,1
24073,42,admin,single,secondary,no,164,yes,yes,unknown,22,oct,1,0.986355,0
24095,30,admin,married,secondary,no,1310,no,no,telephone,27,oct,1,0.984226,0
24088,37,admin,married,secondary,no,1967,no,no,telephone,27,oct,1,0.982595,1
33747,65,retired,married,primary,no,276,no,no,cellular,22,apr,1,0.977599,1
31334,30,management,single,tertiary,no,3473,no,no,cellular,12,mar,2,0.974815,1
31207,36,technician,divorced,tertiary,no,1381,no,no,cellular,2,mar,1,0.974382,0
31066,63,retired,married,tertiary,no,133,yes,no,cellular,13,feb,2,0.973318,0
24086,44,blue-collar,married,secondary,no,1324,yes,no,telephone,25,oct,1,0.972902,0
31281,44,management,single,tertiary,no,483,no,no,cellular,6,mar,2,0.969821,1


In [15]:
# Assign each customer to a priority group
def assign_priority(probability):
    if probability >= 0.70:
        return "High"
    elif probability >= 0.40:
        return "Medium"
    else:
        return "Low"


customer_analysis["priority"] = (customer_analysis["subscription_probability"].apply(assign_priority))
customer_analysis["priority"].value_counts()

priority
Low       5869
Medium    1880
High       251
Name: count, dtype: int64

In [16]:
priority_summary = customer_analysis.groupby("priority").agg(
    customers=("subscription_probability", "count"),
    avg_probability=("subscription_probability", "mean"),
    actual_subscription_rate=("actual_subscription", "mean")
)

priority_summary = priority_summary.sort_values("avg_probability", ascending=False)

priority_summary

,customers,avg_probability,actual_subscription_rate
priority,,,
High,251,0.829279,0.442231
Medium,1880,0.504621,0.110106
Low,5869,0.203239,0.044471


In [17]:
high_priority = customer_analysis[customer_analysis["priority"] == "High"]

categorical_features = [
    "contact",
    "month",
    "job",
    "marital",
    "education",
    "housing",
    "loan",
    "default"
]

for feature in categorical_features:
    print(f"\n--- {feature} ---")
    print(high_priority[feature].value_counts(normalize=True).head(10))


--- contact ---
contact
cellular     0.896414
telephone    0.079681
unknown      0.023904
Name: proportion, dtype: float64

--- month ---
month
apr    0.394422
may    0.183267
feb    0.115538
mar    0.103586
jun    0.071713
oct    0.047809
nov    0.031873
jul    0.031873
aug    0.015936
jan    0.003984
Name: proportion, dtype: float64

--- job ---
job
management       0.266932
technician       0.243028
admin            0.127490
blue-collar      0.087649
retired          0.067729
services         0.059761
unemployed       0.043825
self-employed    0.039841
student          0.035857
entrepreneur     0.011952
Name: proportion, dtype: float64

--- marital ---
marital
married     0.482072
single      0.402390
divorced    0.115538
Name: proportion, dtype: float64

--- education ---
education
tertiary     0.462151
secondary    0.458167
primary      0.047809
missing      0.031873
Name: proportion, dtype: float64

--- housing ---
housing
no     0.657371
yes    0.342629
Name: proportion, dtype:

In [18]:
#compare high priority customers with overall customer base
for feature in categorical_features:
    print(f"\n--- {feature} ---")

    high_percentage = (high_priority[feature].value_counts(normalize=True))

    overall_percentage = (customer_analysis[feature].value_counts(normalize=True))

    comparison = pd.DataFrame({
        "High Priority %": high_percentage,
        "Overall %": overall_percentage
    })

    comparison["Difference"] = (comparison["High Priority %"] - comparison["Overall %"])
    print(comparison.sort_values("Difference", ascending=False))


--- contact ---
           High Priority %  Overall %  Difference
contact                                          
cellular          0.896414   0.617875    0.278539
telephone         0.079681   0.059125    0.020556
unknown           0.023904   0.323000   -0.299096

--- month ---
       High Priority %  Overall %  Difference
month                                        
apr           0.394422   0.062125    0.332297
mar           0.103586   0.005250    0.098336
feb           0.115538   0.063000    0.052538
oct           0.047809   0.001875    0.045934
jan           0.003984   0.030500   -0.026516
jun           0.071713   0.120625   -0.048912
nov           0.031873   0.093375   -0.061502
aug           0.015936   0.129500   -0.113564
jul           0.031873   0.156500   -0.124627
may           0.183267   0.337000   -0.153733
dec                NaN   0.000250         NaN

--- job ---
               High Priority %  Overall %  Difference
job                                                  

In [19]:
#analyze numerical features for high priority customers
numerical_features = [
    "age",
    "balance",
    "day",
    "campaign"
]

numerical_summary = pd.DataFrame({
    "Overall Mean": customer_analysis[numerical_features].mean(),
    "High Priority Mean": high_priority[numerical_features].mean(),
    "Overall Median": customer_analysis[numerical_features].median(),
    "High Priority Median": high_priority[numerical_features].median()
})

numerical_summary

,Overall Mean,High Priority Mean,Overall Median,High Priority Median
age,40.644750,39.509960,39.0,36.0
balance,1275.898125,2282.573705,408.5,1066.0
day,15.937750,16.466135,16.0,17.0
campaign,2.913250,1.872510,2.0,1.0


In [20]:
customer_analysis["age_group"] = pd.cut(customer_analysis["age"], bins=[0, 30, 40, 50, 60, 100],
    labels=["Under 30", "30-39", "40-49", "50-59", "60+"]
)

customer_analysis["balance_group"] = pd.cut(
    customer_analysis["balance"],
    bins=[-float("inf"), 0, 1000, 5000, 10000, float("inf")],
    labels=["Negative", "0-999", "1,000-4,999", "5,000-9,999", "10,000+"]
)

In [21]:
age_segments = customer_analysis.groupby("age_group", observed=True).agg(
    customers=("subscription_probability", "count"),
    avg_probability=("subscription_probability", "mean"),
    actual_subscription_rate=("actual_subscription", "mean")
)

age_segments = age_segments.sort_values("avg_probability", ascending=False)

age_segments

,customers,avg_probability,actual_subscription_rate
age_group,,,
60+,39,0.519832,0.307692
Under 30,1153,0.353556,0.108413
30-39,3245,0.312163,0.069954
40-49,2047,0.269254,0.058134
50-59,1516,0.235878,0.063325


In [22]:
balance_segments = customer_analysis.groupby("balance_group", observed=True).agg(
    customers=("subscription_probability", "count"),
    avg_probability=("subscription_probability", "mean"),
    actual_subscription_rate=("actual_subscription", "mean")
)

balance_segments = balance_segments.sort_values("avg_probability", ascending=False)
balance_segments

,customers,avg_probability,actual_subscription_rate
balance_group,,,
"1,000-4,999",1981,0.338645,0.097930
0-999,4219,0.299051,0.065418
"5,000-9,999",321,0.284727,0.102804
"10,000+",134,0.223983,0.104478
Negative,1345,0.219841,0.046097
